# Tái lập huấn luyện BTS trên NYU Depth V2


## 1. Thiết kế thực nghiệm

| Thành phần | Thiết lập |
|---|---|
| Model | BTS |
| Encoder | DenseNet161, ImageNet pretrained |
| Dataset | NYU Depth V2 |
| Training samples | 24,231 |
| Test samples | 654 |
| Input training | 416 × 544 |
| Global batch size | 4 |
| GPU | 2 × Tesla T4 |
| Epochs | 50 |
| Optimizer | AdamW |
| Initial learning rate | 1e-4 |
| Weight decay | 1e-2 |
| Adam epsilon | 1e-3 |
| Depth range | 1e-3 m – 10 m |
| Data augmentation | Random rotation ±2.5° |
| Online evaluation | Mỗi 500 global steps |
| Evaluation crop | Eigen crop |
| Training loss | Scale-Invariant Logarithmic Loss (SILog) |

### Kiểm thử tiền nghiệm đã hoàn tất

Trước full run, pipeline đã được kiểm tra trên cùng môi trường 2 × T4:

| Kiểm thử | Kết quả |
|---|---|
| DDP training 416×544, global batch 4 | 10/10 steps hoàn thành |
| Rolling recovery checkpoint | `model-latest` hợp lệ |
| Mid-epoch resume | checkpoint step 8 → tiếp tục đúng tại step 9 |
| Optimizer state khi resume | khôi phục thành công |
| Online evaluation | đủ 654 mẫu |
| 9 depth metrics | tính toán và đồng bộ giữa 2 GPU |
| Best-checkpoint tracking | hoạt động |




In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input/bts-nyuv2-full-training")
TEMP = Path("/kaggle/temp")
WORK = Path("/kaggle/working")

MODEL_NAME = "bts_nyu_kaggle_full"

RUN_TRAINING = True
RUN_FINAL_EVALUATION = True

RESUME_CHECKPOINT = ""

TRAIN_EPOCHS = 50
GLOBAL_BATCH_SIZE = 4
INPUT_HEIGHT = 416
INPUT_WIDTH = 544
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
ADAM_EPS = 1e-3
MAX_DEPTH = 10.0
EVAL_FREQ = 500
SAVE_FREQ = 500

print("Model:", MODEL_NAME)
print("Input dataset:", INPUT)
print("Resume checkpoint:", RESUME_CHECKPOINT or "fresh run")


## 3. Môi trường thực thi

Phần này ghi nhận phiên bản Python, PyTorch, CUDA và GPU thực tế của phiên Kaggle. Thông tin được lưu lại trong manifest ở cuối notebook để bảo đảm khả năng truy vết thí nghiệm.


In [ ]:
import sys
import subprocess
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() == 2, (
    "Notebook này được thiết kế cho Kaggle GPU T4 x2."
)

print()
print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
    check=True,
).stdout)


## 4. Chuẩn bị mã nguồn và dataset

Dataset Kaggle gồm bốn thành phần:

- mã nguồn BTS đã kiểm thử;
- NYUv2 synchronized training set;
- official test split;
- DenseNet161 ImageNet weights.

Dataset chỉ được đọc từ `/kaggle/input`. File tạm được đặt trong `/kaggle/temp`, còn checkpoint, log và kết quả cuối được lưu tại `/kaggle/working`.


In [ ]:
import os
import shutil
import zipfile
import hashlib
from pathlib import Path

assert INPUT.exists(), f"Không tìm thấy dataset: {INPUT}"

BTS_ROOT = TEMP / "bts"
NYU_ROOT = TEMP / "dataset" / "nyu_depth_v2"

def remove_runtime_path(path: Path):
    if path.is_symlink():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)

remove_runtime_path(BTS_ROOT)
remove_runtime_path(NYU_ROOT)

BTS_ROOT.mkdir(parents=True, exist_ok=True)
NYU_ROOT.mkdir(parents=True, exist_ok=True)

source_zip = INPUT / "bts_kaggle_source.zip"
source_dir = INPUT / "bts_kaggle_source"

if source_zip.exists():
    with zipfile.ZipFile(source_zip) as z:
        z.extractall(BTS_ROOT)
elif source_dir.exists():
    shutil.copytree(source_dir, BTS_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Không tìm thấy BTS source.")

train_root = NYU_ROOT / "sync"
sync_zip = INPUT / "sync.zip"
nested_sync = INPUT / "sync" / "sync"
flat_sync = INPUT / "sync"

if sync_zip.exists():
    with zipfile.ZipFile(sync_zip) as z:
        z.extractall(NYU_ROOT)
elif nested_sync.exists():
    train_root.symlink_to(nested_sync, target_is_directory=True)
elif flat_sync.exists():
    train_root.symlink_to(flat_sync, target_is_directory=True)
else:
    raise FileNotFoundError("Không tìm thấy NYUv2 sync dataset.")

official_root = NYU_ROOT / "official_splits"
official_root.mkdir(parents=True, exist_ok=True)
test_root = official_root / "test"

test_zip = INPUT / "official_splits_test.zip"
expanded_test = INPUT / "official_splits_test" / "test"

if test_zip.exists():
    with zipfile.ZipFile(test_zip) as z:
        z.extractall(official_root)
elif expanded_test.exists():
    test_root.symlink_to(expanded_test, target_is_directory=True)
else:
    raise FileNotFoundError("Không tìm thấy official NYUv2 test split.")

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("BTS source:", BTS_ROOT)
print("Train root:", train_root)
print("Test root :", test_root)


In [ ]:
weights_src = INPUT / "densenet161-8d451a50.pth"
assert weights_src.exists(), f"Missing pretrained weight: {weights_src}"

weights_hash = sha256_file(weights_src)
assert weights_hash.startswith("8d451a50")

cache_dir = Path(torch.hub.get_dir()) / "checkpoints"
cache_dir.mkdir(parents=True, exist_ok=True)

weights_dst = cache_dir / weights_src.name
shutil.copy2(weights_src, weights_dst)

print("DenseNet161:", weights_dst)
print("SHA256:", weights_hash)
print("Size:", round(weights_dst.stat().st_size / 1024**2, 2), "MB")


In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec("tensorboardX") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "tensorboardX"]
    )

import numpy as np
import pandas as pd
import cv2
import scipy
import h5py
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("SciPy:", scipy.__version__)
print("h5py:", h5py.__version__)
print("tensorboardX: available")


## 5. Kiểm tra tính toàn vẹn dữ liệu

Không bắt đầu training nếu file list và dữ liệu vật lý không khớp. Toàn bộ đường dẫn RGB và depth ground truth của train/test được kiểm tra trước khi khởi tạo model.


In [ ]:
train_list = BTS_ROOT / "train_test_inputs" / "nyudepthv2_train_files_with_gt.txt"
test_list = BTS_ROOT / "train_test_inputs" / "nyudepthv2_test_files_with_gt.txt"

train_rows = [
    line.strip().split()
    for line in train_list.read_text().splitlines()
    if line.strip()
]
test_rows = [
    line.strip().split()
    for line in test_list.read_text().splitlines()
    if line.strip()
]

missing_train_rgb = [
    row[0] for row in train_rows
    if not (train_root / row[0].lstrip("/")).exists()
]
missing_train_gt = [
    row[1] for row in train_rows
    if not (train_root / row[1].lstrip("/")).exists()
]
missing_test_rgb = [
    row[0] for row in test_rows
    if not (test_root / row[0].lstrip("/")).exists()
]
missing_test_gt = [
    row[1] for row in test_rows
    if not (test_root / row[1].lstrip("/")).exists()
]

integrity = pd.DataFrame([
    {
        "Split": "Train",
        "Samples": len(train_rows),
        "Missing RGB": len(missing_train_rgb),
        "Missing depth": len(missing_train_gt),
    },
    {
        "Split": "Test",
        "Samples": len(test_rows),
        "Missing RGB": len(missing_test_rgb),
        "Missing depth": len(missing_test_gt),
    },
])

display(integrity)

assert len(train_rows) == 24231
assert len(test_rows) == 654
assert not missing_train_rgb
assert not missing_train_gt
assert not missing_test_rgb
assert not missing_test_gt

print("Dataset integrity check completed.")


## 6. Tương thích runtime và checkpoint

Mã nguồn BTS được giữ nguyên về kiến trúc, loss, optimizer và protocol đánh giá. Runtime chỉ bổ sung các cơ chế kỹ thuật cần thiết:

- checkpoint loading tương thích PyTorch hiện đại;
- rolling `model-latest` để phục hồi khi Kaggle session bị ngắt;
- resume đúng vị trí giữa epoch;
- `model-final` lưu đúng trạng thái sau optimization step cuối.

Các thay đổi này không thay đổi phép tính forward, SILog loss, AdamW update hay metric đánh giá.


In [ ]:
import re
import subprocess

bts_main_path = BTS_ROOT / "pytorch" / "bts_main.py"
bts_test_path = BTS_ROOT / "pytorch" / "bts_test.py"

assert bts_main_path.exists()
assert bts_test_path.exists()

def ensure_weights_only_false(path: Path):
    text = path.read_text(encoding="utf-8")
    replacements = [
        (
            "torch.load(args.checkpoint_path, map_location=loc)",
            "torch.load(args.checkpoint_path, map_location=loc, weights_only=False)",
        ),
        (
            "torch.load(args.checkpoint_path)",
            "torch.load(args.checkpoint_path, weights_only=False)",
        ),
    ]
    for old, new in replacements:
        if new not in text and old in text:
            text = text.replace(old, new)
    path.write_text(text, encoding="utf-8")

def ensure_mid_epoch_resume(path: Path):
    text = path.read_text(encoding="utf-8")

    markers = [
        "last_completed_step = checkpoint['global_step']",
        "global_step = last_completed_step + 1",
        "resume_step_in_epoch = global_step % steps_per_epoch",
        "if model_just_loaded and step < resume_step_in_epoch:",
    ]
    if all(marker in text for marker in markers):
        return

    old_load = (
        "            global_step = checkpoint['global_step']\n"
        "            model.load_state_dict(checkpoint['model'])"
    )
    new_load = (
        "            last_completed_step = checkpoint['global_step']\n"
        "            global_step = last_completed_step + 1\n"
        "            model.load_state_dict(checkpoint['model'])"
    )
    if old_load not in text:
        raise RuntimeError("Không tìm thấy checkpoint loading block.")
    text = text.replace(old_load, new_load, 1)

    old_print = (
        "            print(\"Loaded checkpoint '{}' (global_step {})\".format("
        "args.checkpoint_path, checkpoint['global_step']))"
    )
    new_print = """            print(
                "Loaded checkpoint '{}' "
                "(last completed global_step {}, "
                "resuming from global_step {})".format(
                    args.checkpoint_path,
                    last_completed_step,
                    global_step
                )
            )"""
    if old_print in text:
        text = text.replace(old_print, new_print, 1)

    epoch_line = "    epoch = global_step // steps_per_epoch"
    if "resume_step_in_epoch = global_step % steps_per_epoch" not in text:
        text = text.replace(
            epoch_line,
            epoch_line + "\n    resume_step_in_epoch = global_step % steps_per_epoch",
            1,
        )

    old_loop = (
        "        for step, sample_batched in enumerate(dataloader.data):\n"
        "            optimizer.zero_grad()"
    )
    new_loop = """        for step, sample_batched in enumerate(dataloader.data):
            if model_just_loaded and step < resume_step_in_epoch:
                continue

            if model_just_loaded:
                print(
                    "Resume position: epoch {}, "
                    "step {}/{}, global_step {}".format(
                        epoch,
                        step,
                        steps_per_epoch,
                        global_step
                    )
                )
                model_just_loaded = False

            optimizer.zero_grad()"""
    if "if model_just_loaded and step < resume_step_in_epoch:" not in text:
        if old_loop not in text:
            raise RuntimeError("Không tìm thấy training-loop insertion point.")
        text = text.replace(old_loop, new_loop, 1)

    path.write_text(text, encoding="utf-8")

FINAL_PATCH_MARKER = "# FINAL_CHECKPOINT_PATCH_V1"

def ensure_final_checkpoint(path: Path):
    text = path.read_text(encoding="utf-8")
    if FINAL_PATCH_MARKER in text:
        return

    anchor = (
        "    if not args.multiprocessing_distributed or "
        "(args.multiprocessing_distributed "
        "and args.rank % ngpus_per_node == 0):\n"
        "        writer.close()"
    )
    if anchor not in text:
        raise RuntimeError("Không tìm thấy writer-close anchor.")

    block = """    # FINAL_CHECKPOINT_PATCH_V1
    if global_step > 0 and (not args.distributed or args.rank == 0):
        final_completed_step = global_step - 1

        final_checkpoint = {
            'global_step': final_completed_step,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_eval_measures_higher_better': best_eval_measures_higher_better,
            'best_eval_measures_lower_better': best_eval_measures_lower_better,
            'best_eval_steps': best_eval_steps
        }

        experiment_dir = os.path.join(
            args.log_directory,
            args.model_name
        )

        final_path = os.path.join(experiment_dir, 'model-final')
        final_tmp = final_path + '.tmp'
        torch.save(final_checkpoint, final_tmp)
        os.replace(final_tmp, final_path)

        latest_path = os.path.join(experiment_dir, 'model-latest')
        latest_tmp = latest_path + '.tmp'
        shutil.copy2(final_path, latest_tmp)
        os.replace(latest_tmp, latest_path)

        print(
            "Saved final checkpoint: {} "
            "(global_step {})".format(
                final_path,
                final_completed_step
            )
        )

"""
    text = text.replace(anchor, block + anchor, 1)
    path.write_text(text, encoding="utf-8")

source_text = bts_main_path.read_text(encoding="utf-8")
assert "model-latest" in source_text
assert "os.replace" in source_text
assert "Saved recovery checkpoint" in source_text

ensure_mid_epoch_resume(bts_main_path)
ensure_final_checkpoint(bts_main_path)
ensure_weights_only_false(bts_main_path)
ensure_weights_only_false(bts_test_path)

for path in [bts_main_path, bts_test_path]:
    result = subprocess.run(
        [sys.executable, "-m", "py_compile", str(path)],
        capture_output=True,
        text=True,
    )
    assert result.returncode == 0, result.stderr

runtime_checks = pd.DataFrame([
    {"Check": "Rolling recovery checkpoint", "Status": "OK"},
    {"Check": "Mid-epoch resume", "Status": "OK"},
    {"Check": "Final checkpoint", "Status": "OK"},
    {"Check": "PyTorch checkpoint loading", "Status": "OK"},
])

display(runtime_checks)
print("Runtime BTS SHA256:", sha256_file(bts_main_path))


## 7. Cấu hình huấn luyện đầy đủ

Với 24,231 mẫu và 2 GPU, `DistributedSampler` phân phối 12,116 mẫu cho mỗi GPU. Batch cục bộ là 2, tương ứng **6,058 optimization steps/epoch** và **302,900 steps cho 50 epochs**.

Learning rate được giữ theo schedule của BTS và online evaluation được thực hiện mỗi 500 global steps.


In [ ]:
import math
import gc
import json

PYTORCH_DIR = BTS_ROOT / "pytorch"
MODELS_DIR = WORK / "models"
LOGS_DIR = WORK / "logs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

def write_config(filename: str, text: str) -> Path:
    path = PYTORCH_DIR / filename
    path.write_text(text.strip() + "\n", encoding="utf-8")
    return path

def checkpoint_summary(path: Path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    best_steps = ckpt.get("best_eval_steps")
    summary = {
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "global_step": int(ckpt["global_step"]),
        "model_tensors": len(ckpt["model"]),
        "optimizer_groups": len(ckpt["optimizer"]["param_groups"]),
        "best_eval_steps": (
            best_steps.tolist()
            if hasattr(best_steps, "tolist")
            else best_steps
        ),
    }
    del ckpt
    gc.collect()
    return summary

def stage_resume_checkpoint(source: Path) -> Path:
    source = Path(source)
    assert source.exists(), source

    stage_dir = TEMP / "resume" / MODEL_NAME
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)

    staged_ckpt = stage_dir / "model-latest"
    shutil.copy2(source, staged_ckpt)

    candidates = [
        source.parent / f"{source.parent.name}.py",
        source.parent / f"{MODEL_NAME}.py",
    ]
    architecture = next((p for p in candidates if p.exists()), None)
    if architecture is None:
        architecture = BTS_ROOT / "pytorch" / "bts.py"

    shutil.copy2(
        architecture,
        stage_dir / f"{MODEL_NAME}.py",
    )
    return staged_ckpt

def find_candidate_checkpoint():
    if RESUME_CHECKPOINT.strip():
        p = Path(RESUME_CHECKPOINT.strip())
        if p.exists() and p.stat().st_size > 0:
            return p

    # 1. Check local working directory
    working_latest = MODELS_DIR / MODEL_NAME / "model-latest"
    if working_latest.exists() and working_latest.stat().st_size > 0:
        return working_latest

    # 2. Check mounted kernel sources (/kaggle/input/**/model-latest)
    for p in Path("/kaggle/input").rglob("model-latest"):
        if p.is_file() and p.stat().st_size > 1024 * 1024:
            return p

    # 3. Check best checkpoints
    best_candidates = [
        p for p in Path("/kaggle/input").rglob("model-*-best_*")
        if p.is_file() and p.stat().st_size > 1024 * 1024
    ]
    if best_candidates:
        return sorted(best_candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]

    return None

candidate = find_candidate_checkpoint()
resume_staged = None

if candidate is not None:
    print(f"Found checkpoint to resume: {candidate}")
    resume_staged = stage_resume_checkpoint(candidate)
    print("Training mode: RESUME")
    display(pd.DataFrame([checkpoint_summary(resume_staged)]))
else:
    print("Training mode: FRESH (No checkpoint found to resume)")


In [ ]:
samples_per_gpu = math.ceil(len(train_rows) / 2)
local_batch_size = GLOBAL_BATCH_SIZE // 2
steps_per_epoch = math.ceil(samples_per_gpu / local_batch_size)
total_steps = steps_per_epoch * TRAIN_EPOCHS
expected_final_step = total_steps - 1

resume_line = (
    f"--checkpoint_path {resume_staged}"
    if resume_staged is not None
    else ""
)

full_config = f"""
--mode train
--model_name {MODEL_NAME}
--encoder densenet161_bts
--dataset nyu

--data_path {train_root}/
--gt_path {train_root}/
--filenames_file {train_list}

--batch_size {GLOBAL_BATCH_SIZE}
--num_epochs {TRAIN_EPOCHS}
--learning_rate {LEARNING_RATE}
--weight_decay {WEIGHT_DECAY}
--adam_eps {ADAM_EPS}
--num_threads 1

--input_height {INPUT_HEIGHT}
--input_width {INPUT_WIDTH}
--max_depth {MAX_DEPTH}

--do_random_rotate
--degree 2.5

--log_directory {MODELS_DIR}/
--log_freq 100
--save_freq {SAVE_FREQ}

--multiprocessing_distributed
--dist_url tcp://127.0.0.1:2350

--do_online_eval
--eval_freq {EVAL_FREQ}

--data_path_eval {test_root}/
--gt_path_eval {test_root}/
--filenames_file_eval {test_list}
--min_depth_eval 1e-3
--max_depth_eval 10
--eval_summary_directory {MODELS_DIR}/eval/
--eigen_crop

{resume_line}
"""

full_cfg = write_config(
    "arguments_train_nyu_kaggle_full.txt",
    full_config,
)

protocol = pd.DataFrame([
    {"Parameter": "Train samples", "Value": len(train_rows)},
    {"Parameter": "Test samples", "Value": len(test_rows)},
    {"Parameter": "Global batch size", "Value": GLOBAL_BATCH_SIZE},
    {"Parameter": "Local batch / GPU", "Value": local_batch_size},
    {"Parameter": "Steps / epoch", "Value": steps_per_epoch},
    {"Parameter": "Epochs", "Value": TRAIN_EPOCHS},
    {"Parameter": "Total optimization steps", "Value": total_steps},
    {"Parameter": "Expected final global_step", "Value": expected_final_step},
])

display(protocol)
print("Training config:", full_cfg)


## 8. Huấn luyện

Toàn bộ stdout/stderr được lưu tại:

`/kaggle/working/logs/bts_full_train.log`

Notebook chỉ hiển thị các mốc training/evaluation quan trọng để tránh tạo output quá lớn. `model-latest` được cập nhật định kỳ và dùng làm recovery checkpoint nếu session bị gián đoạn.


In [ ]:
progress_pattern = re.compile(
    r"\[(\d+)\]\[(\d+)/(\d+)/(\d+)\], "
    r"lr:\s*([0-9.eE+-]+), loss:\s*([0-9.eE+-]+)"
)

def run_training(config_path: Path, log_path: Path, display_every=500):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        sys.executable,
        "-u",
        "bts_main.py",
        config_path.name,
    ]

    print("Command:", " ".join(cmd))
    print("Log:", log_path)

    process = subprocess.Popen(
        cmd,
        cwd=PYTORCH_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    show_next_eval_line = 0

    with log_path.open("w", encoding="utf-8", buffering=1) as log:
        for line in process.stdout:
            log.write(line)

            match = progress_pattern.search(line)
            if match:
                step_in_epoch = int(match.group(2))
                total_in_epoch = int(match.group(3))
                if (
                    step_in_epoch % display_every == 0
                    or step_in_epoch == total_in_epoch - 1
                ):
                    print(line, end="")
                continue

            if "Computing errors for" in line:
                print(line, end="")
                show_next_eval_line = 2
                continue

            if show_next_eval_line > 0:
                print(line, end="")
                show_next_eval_line -= 1
                continue

            important = (
                "Use GPU:",
                "Model Initialized",
                "Loading checkpoint",
                "Loaded checkpoint",
                "Resume position:",
                "New best for",
                "Saved recovery checkpoint:",
                "Saved final checkpoint:",
                "NaN in loss",
                "Traceback",
                "RuntimeError",
            )
            if any(token in line for token in important):
                print(line, end="")

    returncode = process.wait()
    print("Return code:", returncode)

    if returncode != 0:
        tail = log_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()[-100:]
        print("\n".join(tail))
        raise RuntimeError(
            f"Training failed with return code {returncode}"
        )

    return log_path


In [ ]:
FULL_MODEL_DIR = MODELS_DIR / MODEL_NAME
FULL_TRAIN_LOG = LOGS_DIR / "bts_full_train.log"

if RUN_TRAINING:
    if resume_staged is None and FULL_MODEL_DIR.exists():
        print("Removing stale fresh-run directory:", FULL_MODEL_DIR)
        shutil.rmtree(FULL_MODEL_DIR)

    run_training(
        full_cfg,
        FULL_TRAIN_LOG,
        display_every=500,
    )

    model_final = FULL_MODEL_DIR / "model-final"
    assert model_final.exists(), (
        "Training completed but model-final was not created."
    )

    final_checkpoint_info = checkpoint_summary(model_final)
    display(pd.DataFrame([final_checkpoint_info]))

    assert final_checkpoint_info["global_step"] == expected_final_step
    print("Full 50-epoch training completed.")
else:
    print("Training skipped.")


## 9. Diễn biến huấn luyện

Hai đồ thị dưới đây được sinh trực tiếp từ log của full run:

- SILog training loss theo global step;
- learning-rate schedule theo global step.

Đường trung bình trượt của loss được dùng để quan sát xu hướng tổng thể vì loss theo mini-batch có dao động tự nhiên.


In [ ]:
def parse_training_log(log_path: Path) -> pd.DataFrame:
    rows = []
    if not log_path.exists():
        return pd.DataFrame()

    for line in log_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines():
        m = progress_pattern.search(line)
        if not m:
            continue

        rows.append({
            "epoch": int(m.group(1)),
            "step_in_epoch": int(m.group(2)),
            "steps_per_epoch": int(m.group(3)),
            "global_step": int(m.group(4)),
            "learning_rate": float(m.group(5)),
            "loss": float(m.group(6)),
        })

    return pd.DataFrame(rows)

history = parse_training_log(FULL_TRAIN_LOG)

if history.empty:
    print("Training history is not available yet.")
else:
    history.to_csv(
        WORK / "bts_nyuv2_training_history.csv",
        index=False,
    )

    history["loss_rolling_500"] = (
        history["loss"]
        .rolling(500, min_periods=1)
        .mean()
    )

    display(history.tail())

    plt.figure(figsize=(10, 5))
    plt.plot(history["global_step"], history["loss"], alpha=0.2, label="Batch loss")
    plt.plot(
        history["global_step"],
        history["loss_rolling_500"],
        label="Rolling mean (500 steps)",
    )
    plt.xlabel("Global step")
    plt.ylabel("SILog loss")
    plt.title("BTS training loss on NYU Depth V2")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(history["global_step"], history["learning_rate"])
    plt.xlabel("Global step")
    plt.ylabel("Learning rate")
    plt.title("Learning-rate schedule")
    plt.grid(alpha=0.2)
    plt.show()


## 10. Suy luận và đánh giá cuối cùng

Sau khi full training hoàn tất, `model-final` được dùng để dự đoán depth cho toàn bộ 654 ảnh test. Metric được tính bằng script đánh giá của BTS với NYUv2 Eigen crop.

Các metric báo cáo:

- δ1, δ2, δ3;
- AbsRel;
- SqRel;
- RMSE;
- RMSElog;
- SILog;
- log10.


In [ ]:
FINAL_CHECKPOINT = FULL_MODEL_DIR / "model-final"
FINAL_RESULT_DIR = WORK / f"result_{MODEL_NAME}"
FINAL_TEST_LOG = LOGS_DIR / "bts_final_test.log"
FINAL_EVAL_LOG = LOGS_DIR / "bts_final_eval.log"

test_config = f"""
--encoder densenet161_bts
--data_path {test_root}/
--dataset nyu
--filenames_file {test_list}
--model_name {MODEL_NAME}
--checkpoint_path {FINAL_CHECKPOINT}
--input_height 480
--input_width 640
--max_depth 10
"""

final_test_cfg = write_config(
    "arguments_test_nyu_kaggle_final.txt",
    test_config,
)

if RUN_FINAL_EVALUATION:
    assert FINAL_CHECKPOINT.exists(), (
        "model-final chưa tồn tại. Hoàn tất training trước."
    )

    if FINAL_RESULT_DIR.exists():
        shutil.rmtree(FINAL_RESULT_DIR)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    with FINAL_TEST_LOG.open("w", encoding="utf-8") as log:
        result = subprocess.run(
            [
                sys.executable,
                "-u",
                str(PYTORCH_DIR / "bts_test.py"),
                str(final_test_cfg),
            ],
            cwd=WORK,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
        )

    if result.returncode != 0:
        print(FINAL_TEST_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        )[-8000:])
    assert result.returncode == 0

    raw_dir = FINAL_RESULT_DIR / "raw"
    raw_count = len(list(raw_dir.glob("*.png")))

    print("Prediction PNGs:", raw_count)
    assert raw_count == 654

    eval_result = subprocess.run(
        [
            sys.executable,
            str(BTS_ROOT / "utils" / "eval_with_pngs.py"),
            "--pred_path", str(raw_dir),
            "--gt_path", str(test_root),
            "--dataset", "nyu",
            "--eigen_crop",
            "--min_depth_eval", "1e-3",
            "--max_depth_eval", "10",
        ],
        cwd=WORK,
        capture_output=True,
        text=True,
    )

    FINAL_EVAL_LOG.write_text(
        eval_result.stdout + "\n" + eval_result.stderr,
        encoding="utf-8",
    )

    print(eval_result.stdout)
    assert eval_result.returncode == 0
    assert "Evaluating 654 files" in eval_result.stdout

    print("Final evaluation completed.")
else:
    print("Final evaluation skipped.")


## 11. Kết quả định lượng

Bảng được tạo trực tiếp từ output của `eval_with_pngs.py`.

Hàng **Author checkpoint reproduction** là kết quả sanity-check đã chạy trước đó với checkpoint phát hành bởi tác giả. Hàng **Our 50-epoch run** là kết quả của model được huấn luyện trong notebook này.


In [ ]:
METRIC_NAMES = [
    "d1", "d2", "d3",
    "AbsRel", "SqRel", "RMSE",
    "RMSElog", "SILog", "log10",
]

REFERENCE_METRICS = {
    "d1": 0.885,
    "d2": 0.978,
    "d3": 0.994,
    "AbsRel": 0.110,
    "SqRel": 0.066,
    "RMSE": 0.392,
    "RMSElog": 0.142,
    "SILog": 11.535,
    "log10": 0.047,
}

def parse_final_metrics(log_path: Path):
    if not log_path.exists():
        return None

    lines = log_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    for i, line in enumerate(lines):
        if (
            "d1" in line
            and "AbsRel" in line
            and "SILog" in line
            and i + 1 < len(lines)
        ):
            values = [
                float(x.strip())
                for x in lines[i + 1].split(",")
            ]
            if len(values) == 9:
                return dict(zip(METRIC_NAMES, values))

    return None

final_metrics = parse_final_metrics(FINAL_EVAL_LOG)

comparison_rows = [
    {
        "Experiment": "Author checkpoint reproduction",
        **REFERENCE_METRICS,
    }
]

if final_metrics is not None:
    comparison_rows.append({
        "Experiment": "Our 50-epoch run",
        **final_metrics,
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

comparison.to_csv(
    WORK / "bts_nyuv2_final_metrics.csv",
    index=False,
)

if final_metrics is None:
    print("Final metrics will appear after final evaluation.")


## 12. Ví dụ định tính

Các ví dụ dưới đây lấy trực tiếp từ official test split. Mỗi mẫu gồm ảnh RGB đầu vào, depth ground truth và depth dự đoán của `model-final`.


In [ ]:
def prediction_name_from_row(row):
    rgb_rel = row[0].lstrip("/")
    parts = rgb_rel.split("/")
    scene = parts[0]
    rgb_name = Path(parts[-1]).with_suffix(".png").name
    return f"{scene}_{rgb_name}"

def show_depth_example(index: int):
    row = test_rows[index]

    rgb_path = test_root / row[0].lstrip("/")
    gt_path = test_root / row[1].lstrip("/")
    pred_path = FINAL_RESULT_DIR / "raw" / prediction_name_from_row(row)

    assert rgb_path.exists()
    assert gt_path.exists()
    assert pred_path.exists()

    rgb = cv2.cvtColor(
        cv2.imread(str(rgb_path)),
        cv2.COLOR_BGR2RGB,
    )
    gt = cv2.imread(str(gt_path), -1).astype(np.float32) / 1000.0
    pred = cv2.imread(str(pred_path), -1).astype(np.float32) / 1000.0

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("RGB")
    axes[0].axis("off")

    axes[1].imshow(gt, vmin=0, vmax=MAX_DEPTH)
    axes[1].set_title("Ground truth depth (m)")
    axes[1].axis("off")

    axes[2].imshow(pred, vmin=0, vmax=MAX_DEPTH)
    axes[2].set_title("Predicted depth (m)")
    axes[2].axis("off")

    fig.suptitle(f"NYUv2 test sample #{index}")
    plt.tight_layout()
    plt.show()

if RUN_FINAL_EVALUATION and FINAL_RESULT_DIR.exists():
    for idx in [0, len(test_rows) // 2, len(test_rows) - 1]:
        show_depth_example(idx)
else:
    print("Qualitative examples will appear after final evaluation.")


## 13. Tổng hợp thí nghiệm và lưu thông tin tái lập

Manifest cuối cùng lưu môi trường phần mềm, GPU, kích thước dataset, SHA256 của source/runtime, training protocol, checkpoint và metric cuối.


In [ ]:
import datetime
import platform

manifest = {
    "created_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "experiment": "BTS NYUv2 full baseline",
    "model": "BTS",
    "encoder": "DenseNet161",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ],
    "train_samples": len(train_rows),
    "test_samples": len(test_rows),
    "input_size": [INPUT_HEIGHT, INPUT_WIDTH],
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "epochs": TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "adam_eps": ADAM_EPS,
    "max_depth": MAX_DEPTH,
    "eval_freq": EVAL_FREQ,
    "save_freq": SAVE_FREQ,
    "steps_per_epoch": steps_per_epoch,
    "total_steps": total_steps,
    "expected_final_global_step": expected_final_step,
    "bts_main_sha256_runtime": sha256_file(bts_main_path),
    "densenet161_sha256": weights_hash,
    "resume_checkpoint_requested": RESUME_CHECKPOINT,
}

model_latest = FULL_MODEL_DIR / "model-latest"
model_final = FULL_MODEL_DIR / "model-final"

if model_latest.exists():
    manifest["model_latest"] = checkpoint_summary(model_latest)

if model_final.exists():
    manifest["model_final"] = checkpoint_summary(model_final)

if final_metrics is not None:
    manifest["final_metrics"] = final_metrics

manifest_path = WORK / "bts_nyuv2_experiment_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Manifest:", manifest_path)
display(pd.DataFrame([
    {"Artifact": "Training log", "Path": str(FULL_TRAIN_LOG)},
    {"Artifact": "Training history", "Path": str(WORK / "bts_nyuv2_training_history.csv")},
    {"Artifact": "Final metrics", "Path": str(WORK / "bts_nyuv2_final_metrics.csv")},
    {"Artifact": "Manifest", "Path": str(manifest_path)},
    {"Artifact": "Final checkpoint", "Path": str(model_final)},
]))


## 14. Kết luận baseline

Baseline BTS được xem là hoàn thành khi thỏa đồng thời:

- full 50 epochs kết thúc;
- `model-final` có `global_step = 302899`;
- 654 prediction PNG được sinh;
- final evaluation chạy đủ 654 mẫu;
- bảng 9 metric được tạo;
- training history, log và manifest được lưu.

Baseline này là mốc tham chiếu cho các thí nghiệm tiếp theo. Mọi cải tiến mô hình nên giữ cố định split dữ liệu, depth range, evaluation crop và metric để bảo đảm so sánh công bằng.

### Tài liệu tham khảo

1. Jin Han Lee et al., *From Big to Small: Multi-Scale Local Planar Guidance for Monocular Depth Estimation*.
2. BTS official repository: `https://github.com/cleinc/bts`
3. NYU Depth V2 dataset.
